# Tugas 1 - Milestone 3: Analisis Temporal dan Advanced EDA

**Mata Kuliah:** Analisis Big Data  
**Nama Mahasiswa:** Mohammad Ravlindo Saputra  
**NIM:** 202310370311115  
**Username GitHub:** Ravlindo

Notebook ini menggunakan dataset bersih dari Milestone 2 untuk menganalisis pola publikasi berita Indonesia. Pengerjaan dibagi menjadi tiga tahap:

1. inisialisasi dan analisis temporal/window;
2. enam visualisasi interaktif;
3. lima insight analitik dan validasi akhir.

## Bagian 1: Inisialisasi dan Memuat Data Bersih

In [1]:
from pathlib import Path

import duckdb
import plotly.express as px
import plotly.graph_objects as go
import polars as pl

pl.Config.set_tbl_rows(20)
pl.Config.set_tbl_cols(20)

print(f"Polars {pl.__version__} | DuckDB {duckdb.__version__}")

Polars 1.44.2 | DuckDB 1.5.6


In [2]:
cwd = Path.cwd().resolve()
if (cwd / "data").exists():
    PROJECT_ROOT = cwd
elif (cwd.parent / "data").exists():
    PROJECT_ROOT = cwd.parent
else:
    raise FileNotFoundError("Root project tidak ditemukan.")

DATA_PATH = PROJECT_ROOT / "data" / "processed" / "dataset_bersih.parquet"
if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Dataset bersih tidak ditemukan: {DATA_PATH}. "
        "Jalankan 02_data_cleaning.ipynb terlebih dahulu."
    )

print(f"Dataset: {DATA_PATH}")
print(f"Ukuran file: {DATA_PATH.stat().st_size / (1024 ** 2):,.2f} MB")

Dataset: C:\Users\endo1\Documents\SEMESTER  7\big data\tugas1-MohammadRavlindoS\data\processed\dataset_bersih.parquet
Ukuran file: 691.27 MB


In [3]:
df = pl.scan_parquet(DATA_PATH)
schema = df.collect_schema()
required_columns = {
    "date", "link", "title", "text", "source_domain",
    "title_length", "text_length",
}
missing_columns = required_columns - set(schema.names())
if missing_columns:
    raise KeyError(f"Kolom wajib tidak ditemukan: {sorted(missing_columns)}")

data_summary = df.select(
    pl.len().alias("jumlah_baris"),
    pl.col("date").min().alias("tanggal_awal"),
    pl.col("date").max().alias("tanggal_akhir"),
    pl.col("source_domain").n_unique().alias("jumlah_domain"),
).collect()

print(f"Data loaded: {data_summary.item(0, 'jumlah_baris'):,} baris x {len(schema)} kolom")
data_summary

Data loaded: 928,082 baris x 11 kolom


jumlah_baris,tanggal_awal,tanggal_akhir,jumlah_domain
u32,date,date,u32
928082,2019-05-10,2023-10-01,175


In [4]:
df.head(5).collect()

date,link,title,text,title_length,text_length,title_length_outlier,text_length_outlier,title_length_clean,text_length_clean,source_domain
date,str,str,str,u32,u32,bool,bool,u32,u32,str
2019-05-10,"""https://www.cnbcindonesia.com/…","""Potret Jemaah Haji Mulai Melak…","""Haji tahun ini merupakan tanta…",57,438,false,false,57,438,"""cnbcindonesia.com"""
2019-08-19,"""https://www.cnbcindonesia.com/…","""Cek Warga China Mencoba Robot …","""Seorang anggota staf mengambil…",61,460,false,false,61,460,"""cnbcindonesia.com"""
2019-08-26,"""https://www.cnbcindonesia.com/…","""Ajaib! Tato Elektronik Buatan …","""""Di masa depan, apa yang kami …",58,327,false,false,58,327,"""cnbcindonesia.com"""
2019-10-30,"""https://www.cnbcindonesia.com/…","""Chairul Tanjung Hadiri BSI GIF…","""Menurutnya, karakter anak-anak…",36,601,false,false,36,601,"""cnbcindonesia.com"""
2022-03-28,"""https://www.cnbcindonesia.com/…","""11 Tanda Rumah Anda Sudah Terc…","""Jakarta, CNBC Indonesia - Polu…",60,2114,false,false,60,2114,"""cnbcindonesia.com"""


## Bagian 2: Analisis Temporal dan Window Functions

Artikel diagregasi per bulan menggunakan `group_by_dynamic()`. Selanjutnya digunakan `rolling_mean()` untuk meredam fluktuasi jangka pendek dan `shift()` untuk menghitung perubahan jumlah artikel dibandingkan bulan sebelumnya.

In [5]:
monthly_lf = (
    df
    .sort("date")
    .group_by_dynamic("date", every="1mo")
    .agg(
        pl.len().cast(pl.Int64).alias("jumlah_artikel"),
        pl.col("source_domain").n_unique().alias("domain_aktif"),
        pl.col("title_length").mean().round(2).alias("rata_panjang_judul"),
        pl.col("text_length").mean().round(2).alias("rata_panjang_teks"),
    )
    .sort("date")
)

monthly = (
    monthly_lf
    .with_columns(
        pl.col("jumlah_artikel").rolling_mean(window_size=3).alias("moving_average_3_bulan"),
        pl.col("jumlah_artikel").shift(1).alias("artikel_bulan_sebelumnya"),
    )
    .with_columns(
        ((pl.col("jumlah_artikel") - pl.col("artikel_bulan_sebelumnya"))
         / pl.col("artikel_bulan_sebelumnya") * 100)
        .round(2)
        .alias("pertumbuhan_persen")
    )
    .collect()
)
monthly

date,jumlah_artikel,domain_aktif,rata_panjang_judul,rata_panjang_teks,moving_average_3_bulan,artikel_bulan_sebelumnya,pertumbuhan_persen
date,i64,u32,f64,f64,f64,i64,f64
2019-05-01,1,1,57.0,438.0,null,null,null
2019-08-01,2,1,59.5,393.5,null,1,100.0
2019-10-01,1,1,36.0,601.0,1.333333,2,-50.0
2022-03-01,1,1,60.0,2114.0,1.333333,1,0.0
2022-04-01,3,1,57.67,3096.67,1.666667,1,200.0
2022-06-01,3,1,57.67,1721.33,2.333333,3,0.0
2022-07-01,4,1,58.25,3828.75,3.333333,3,33.33
2022-08-01,6295,1,55.44,2238.19,2100.666667,4,157275.0
2022-09-01,9545,42,58.75,2184.96,5281.333333,6295,51.63


### Pola Hari dalam Minggu

Agregasi berikut menyiapkan pola musiman mingguan yang akan digunakan pada visualisasi tahap kedua.

In [6]:
nama_hari = {
    1: "Senin", 2: "Selasa", 3: "Rabu", 4: "Kamis",
    5: "Jumat", 6: "Sabtu", 7: "Minggu",
}

weekday = (
    df
    .with_columns(pl.col("date").dt.weekday().alias("nomor_hari"))
    .group_by("nomor_hari")
    .agg(
        pl.len().alias("jumlah_artikel"),
        pl.col("text_length").mean().round(2).alias("rata_panjang_teks"),
    )
    .sort("nomor_hari")
    .collect()
    .with_columns(
        pl.col("nomor_hari").replace_strict(nama_hari).alias("hari")
    )
)
weekday

nomor_hari,jumlah_artikel,rata_panjang_teks,hari
i8,u32,f64,str
1,150963,2322.99,"""Senin"""
2,155745,2328.44,"""Selasa"""
3,152386,2325.47,"""Rabu"""
4,149673,2320.97,"""Kamis"""
5,142436,2318.51,"""Jumat"""
6,92268,2366.73,"""Sabtu"""
7,84611,2384.4,"""Minggu"""


### Catatan Tahap 1

Tahap pertama menghasilkan tabel bulanan dengan moving average tiga bulan dan pertumbuhan terhadap bulan sebelumnya. Agregasi hari dalam minggu juga telah disiapkan untuk analisis musiman pada tahap visualisasi.